# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', None)}\n")

## 2. Data Overview
Review available record sets, fields, and column IDs in the dataset.

Let's list all record sets and their associated fields and columns, referencing each item using their `@id`.

In [ ]:
# List available record sets with their @id
record_sets = list(dataset.record_sets)
print(f"Record Sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {getattr(rs, 'name', '')}")
    print(f"  RecordSet @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields:")
        for fld in rs.fields:
            print(f"    - {getattr(fld, 'name', '')} (@id: {fld.id}) | Data type: {getattr(fld, 'data_type', None)}")
    if hasattr(rs, 'columns') and rs.columns:
        print(f"  Columns:")
        for col in rs.columns:
            print(f"    - {getattr(col, 'name', '')} (@id: {col.id}) | Data type: {getattr(col, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# The @id is used here for referencing. Replace the record_set @ids below with actual @ids (see previous cell's output if not already given).record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show available columns for the first record set
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Columns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Choose a numeric field by its `@id`. We'll use the first numeric column from the main record set by default.

In [ ]:
# List available numeric fields in the main record set
main_df = dataframes[main_record_set_id]

numeric_field_id = None

# Try to infer a numeric field by ID (looking for float/integer columns)
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        possible_numeric_ids = []
        if hasattr(rs, 'fields') and rs.fields:
            for fld in rs.fields:
                dt = getattr(fld, 'data_type', '').lower() if getattr(fld, 'data_type', None) else ''
                if dt in ['float', 'integer', 'number']:
                    possible_numeric_ids.append(fld.id)
        if not possible_numeric_ids and hasattr(rs, 'columns') and rs.columns:
            for col in rs.columns:
                dt = getattr(col, 'data_type', '').lower() if getattr(col, 'data_type', None) else ''
                if dt in ['float', 'integer', 'number']:
                    possible_numeric_ids.append(col.id)
        if possible_numeric_ids:
            numeric_field_id = possible_numeric_ids[0]
            print(f"Numeric field chosen: {numeric_field_id}")
        break

# If we couldn't find a numeric field, default to the first column numerically convertible
if not numeric_field_id:
    for col in main_df.columns:
        try:
            pd.to_numeric(main_df[col])
            numeric_field_id = col
            print(f"Fallback numeric field: {numeric_field_id}")
            break
        except:
            continue

# Convert column to numeric (if possible)
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 0
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df[[numeric_field_id]].head())

filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a categorical field (e.g. one with 'desc', 'group', or 'sample' in the name)
possible_group_fields = [col for col in main_df.columns if 
                        any(x in col.lower() for x in ['desc', 'group', 'sample', 'cat']) and 
                        main_df[col].nunique() < 20]
group_field = possible_group_fields[0] if possible_group_fields else None

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot a histogram of the selected numeric field after filtering, and optionally a boxplot grouped by a categorical variable if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Boxplot by group, if group_field exists
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
    plt.title(f'{numeric_field_id} by {group_field}')
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the Croissant-based dataset and examined its metadata
- Explored record sets, fields, and referenced them by `@id`
- Loaded tabular data into DataFrames and applied filtering/normalization
- Performed basic EDA and visualization

The dataset is ready for more advanced statistical/proteomic or bioinformatics analyses, including but not limited to comparing protein abundance, investigating sample groupings, or identifying outliers for further investigation.

Be sure to consult the FAIR\(^2\) dataset documentation for schema details and to interpret column values in biological context.